In [1]:
# Built-in library
import json
import re
import warnings
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Generator

import msgspec

# Standard imports
import numpy as np
import pandas as pd
import polars as pl

# Visualization
# import matplotlib.pyplot as plt

# NumPy settings
np.set_printoptions(precision=4)

# Pandas settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 1_000
pd.options.display.max_colwidth = 600

# Polars settings
pl.Config.set_fmt_str_lengths(1_000)
pl.Config.set_tbl_cols(n=1_000)
pl.Config.set_tbl_rows(n=200)

warnings.filterwarnings("ignore")

# Black code formatter (Optional)
%load_ext lab_black

# auto reload imports
%load_ext autoreload
%autoreload 2

In [ ]:
import os

from rich.console import Console
from rich.theme import Theme

custom_theme = Theme(
    {
        "white": "#FFFFFF",  # Bright white
        "info": "#00FF00",  # right green
        "warning": "#FFD700",  # Bright gold
        "error": "#FF1493",  # Deep pink
        "success": "#00FFFF",  # Cyan
        "highlight": "#FF4500",  # Orange-red
    }
)
console = Console(theme=custom_theme)


def create_path(path: str | Path) -> None:
    """
    Create parent directories for the given path if they don't exist.

    Parameters
    ----------
    path : str | Path
        The file path for which to create parent directories.
    """
    # Convert to Path object if it's a string
    path_obj: Path = Path(path) if isinstance(path, str) else path

    # Get the parent directory and create it if it doesn't exist
    path_obj.parent.mkdir(parents=True, exist_ok=True)


def go_up_from_current_directory(*, go_up: int = 1) -> None:
    """This is used to up a number of directories.

    Params:
    -------
    go_up: int, default=1
        This indicates the number of times to go back up from the current directory.

    Returns:
    --------
    None
    """
    import sys

    CONST: str = "../"
    NUM: str = CONST * go_up

    # Goto the previous directory
    prev_directory = os.path.join(os.path.dirname(__name__), NUM)
    # Get the 'absolute path' of the previous directory
    abs_path_prev_directory = os.path.abspath(prev_directory)

    # Add the path to the System paths
    sys.path.insert(0, abs_path_prev_directory)
    print(abs_path_prev_directory)


# Prevents ruff from complaining about unused imports
_ = (
    pd,
    pl,
    np,
    json,
    msgspec,
    dataclass,
    field,
    Any,
    Path,
    Generator,
    re,
)

## Basic Chat Chain

In [3]:
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

go_up_from_current_directory(go_up=3)

/Users/mac/Desktop/Projects/RAG-Tutorials


In [36]:
import asyncio

from pydantic import BaseModel, Field


class ModelResponse(BaseModel):
    content: str = Field(
        description="A concise answer to the user query not more than 2 - 4 sentences."
    )
    rationale: str = Field(description="The reasoning behind the answer.")

In [ ]:
model = init_chat_model("openrouter:z-ai/glm-4.7-flash").with_structured_output(ModelResponse)

sys_msg = SystemMessage(content="You are a helpful assistant that answers user queries concisely.")
human_msg = HumanMessage(content="Why do parrots talk?")

# Invoke the model
response = model.invoke([sys_msg, human_msg])
console.print(response, style="info")

ModelResponse(
    content="Parrots talk primarily because they're naturally vocal mimics with highly social brains. Their ability
to replicate sounds they hear repeatedly is a survival skill—many bird species use mimicry to learn about their 
environment, identify predators, or communicate with flock members. Some parrots also associate words with objects,
actions, or emotions, forming meaningful connections like dogs learn commands. However, their 'talking' is more 
sophisticated imitation than true language comprehension.",
    rationale="This response explains the biological and evolutionary reasons for parrot vocal mimicry, notes that 
it's imitation rather than actual language, and touches on how some parrots may develop associations with words, 
providing a comprehensive but concise answer."
)

In [31]:
model = init_chat_model(
    "llama3.2-3b",
    model_provider="openai",
    base_url="http://127.0.0.1:1234/v1",
    api_key="not-needed",
).with_structured_output(ModelResponse)

sys_msg = SystemMessage(
    content="You are a helpful assistant that answers user queries concisely."
)
human_msg = HumanMessage(content="Why do parrots talk?")

# Invoke the model
response = model.invoke([sys_msg, human_msg])
console.print(response, style="info")

ModelResponse(
    content="Parrots 'talk' due to unique physical and behavioral adaptations.",
    rationale="1. Large syrinx: Parrots have a large syrinx, located at the base of their trachea, which is 
responsible for producing sound in birds. This syrinx is more complex and flexible than that of other birds, 
allowing parrots to produce a wider range of sounds.\n2. Strong syrinx muscles: The syrinx muscles are highly 
developed in parrots, enabling them to vibrate the syrinx rapidly and create different sounds.\n3. Brain structure:
Research suggests that parrots have a more complex brain structure than other birds, which may contribute to their 
ability to mimic human speech and understand context.\n4. Social behavior: Many parrot species are social animals, 
living in flocks or family groups, and communication is essential for their survival. The ability to 'talk' helps 
them maintain social bonds and communicate with each other."
)

In [37]:
from typing import cast

semaphore = asyncio.Semaphore(5)  # Limit to 5 concurrent requests


async def aget_response(message: str) -> ModelResponse:
    async with semaphore:
        human_msg = HumanMessage(content=message)
        response = await model.ainvoke([sys_msg, human_msg])
        return cast(ModelResponse, response)


async def abatch_response(messages: list[str]) -> list[ModelResponse]:
    tasks = [aget_response(message) for message in messages]
    return await asyncio.gather(*tasks)


batch_response = await abatch_response(
    [
        "Why do parrots talk?",
        "What is the capital of France?",
        "What are the different car engine types? v6, v8???",
    ]
)


console.print(batch_response)

[
    ModelResponse(
        content='parrots can mimic sounds and phrases due to their unique syrinx anatomy, vocal learning abilities,
and social behavior.',
        rationale='Syntexis Syrinx: Parrots have a different organ called the syrinx located at the base of the 
trachea. This flexible vocal organ is capable of producing a wide range of sounds. Their syrinx can be stretched 
more than those in mammals and birds, enabling them to produce various frequencies.'
    ),
    ModelResponse(content='Paris', rationale='The capital city of France.'),
    ModelResponse(
        content='There are several main types of car engines, including:',
        rationale='1. Inline Engines (e.g., inline-4 or inline-6):\\   - Have cylinders aligned in a straight 
line.\\   - Often used in smaller cars, sports cars, and hybrid vehicles.\\ 2. V-Type Engines (e.g., V6 or V8):\\  
- Have cylinders arranged in a '
    )
]

neidu
